In [1]:
# Cài nnU-Net bản chuẩn từ PyPI (Nhanh và sạch)
!pip install -q nnunetv2

# Cài thêm các thư viện bổ trợ nếu thiếu
!pip install -q nibabel matplotlib medpy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.9/212.9 kB 8.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!nvidia-smi

Tue Mar  3 01:27:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# A. Set up

## 1. Giải nén `nnUNet_raw` & `nnUNet_preprocessed`

In [3]:
import os
import zipfile
from tqdm import tqdm

RAW_ZIP = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_raw/Dataset101_BraTS2020.zip"
PRE_ZIP = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_preprocessed/Dataset101_BraTS2020.zip"

RAW_DIR = "/content"
PRE_DIR = "/content"

def unzip(zip_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        for f in tqdm(z.namelist()):
            z.extract(f, out_dir)

In [4]:
unzip(RAW_ZIP, RAW_DIR)

100%|██████████| 1848/1848 [02:08<00:00, 14.35it/s]


In [5]:
unzip(PRE_ZIP, PRE_DIR)

100%|██████████| 2590/2590 [04:02<00:00, 10.69it/s]


## 2. Set biến môi trường nnU-Net v2

In [6]:
import os

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])

nnUNet_raw: /content/nnUNet_raw
nnUNet_preprocessed: /content/nnUNet_preprocessed
nnUNet_results: /content/nnUNet_results


# B. Inference Uncertainty Map

In [7]:
# CHẾ ĐỘ 1: CHẠY MODEL EDL (UNCERTAINTY DECOMPOSITION)
# ------------------------------------------------------------------
# Script này sẽ thực hiện quy trình full:
# 1. Tự động "tiêm" (inject) file EDLTrainer.py vào thư viện nnU-Net hệ thống.
# 2. Load trọng số Model EDL (checkpoint_best.pth) từ đường dẫn config 'edl'.
# 3. Chạy Inference và tính toán toán học để tách Uncertainty thành:
#    - Aleatoric (Nhiễu dữ liệu)
#    - Epistemic (Mô hình chưa biết)
# 4. Lưu kết quả Segmentation + 3 bản đồ Uncertainty (.nii.gz).

In [8]:
# Đổi tên file cho đúng chuẩn (giải nén còn thừa .gz)
!for f in /content/nnUNet_raw/Dataset101_BraTS2020/imagesTr/*.nii.gz; do mv "$f" "${f%.gz}"; done

## 1. Fold 0

In [9]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode edl_raw --fold 0 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: EDL_RAW | FOLD: 0 ---
🔧 Initializing Engine | Mode: EDL...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=False | Bước trượt=1.0
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 0: 74 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 74 cases (Filtered).

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_011...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_011.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_011.nii -> KHÔNG
100% 8/8 [00:02<00:00,  3.63it/s]
1        | BRATS_011       | 0.9245   | 0.9052   | 0.7905   | 0.8734
    📸 Drawing Slice: 112
    ✅ Saved Snapshot: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Dataset101_BraTS2020/EDLTr

## 2. Fold 1

In [10]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode edl_raw --fold 1 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: EDL_RAW | FOLD: 1 ---
🔧 Initializing Engine | Mode: EDL...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=False | Bước trượt=1.0
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 1: 74 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 74 cases (Filtered).

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_004...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_004.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_004.nii -> KHÔNG
100% 8/8 [00:01<00:00,  5.71it/s]
1        | BRATS_004       | 0.9513   | 0.9131   | 0.8830   | 0.9158
    📸 Drawing Slice: 99
    ✅ Saved Snapshot: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Dataset101_BraTS2020/EDLTra

## 3. Fold 2

In [11]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode edl_raw --fold 2 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: EDL_RAW | FOLD: 2 ---
🔧 Initializing Engine | Mode: EDL...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=False | Bước trượt=1.0
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 2: 74 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 74 cases (Filtered).

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_002...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_002.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_002.nii -> KHÔNG
100% 8/8 [00:01<00:00,  5.66it/s]
1        | BRATS_002       | 0.9075   | 0.9625   | 0.8495   | 0.9065
    📸 Drawing Slice: 35
    ✅ Saved Snapshot: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Dataset101_BraTS2020/EDLTra

## 4. Fold 3

In [12]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode edl_raw --fold 3 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: EDL_RAW | FOLD: 3 ---
🔧 Initializing Engine | Mode: EDL...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=False | Bước trượt=1.0
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 3: 74 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 74 cases (Filtered).

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_003...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_003.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_003.nii -> KHÔNG
100% 8/8 [00:01<00:00,  5.60it/s]
1        | BRATS_003       | 0.8038   | 0.9095   | 0.8505   | 0.8546
    📸 Drawing Slice: 95
    ✅ Saved Snapshot: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Dataset101_BraTS2020/EDLTra

## 5. Fold 4

In [13]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/main.py" --mode edl_raw --fold 4 --run_mode validation_split

🏁 --- STARTING PIPELINE | MODE: EDL_RAW | FOLD: 4 ---
🔧 Initializing Engine | Mode: EDL...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=False | Bước trượt=1.0
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer__nnUNetPlans__3d_fullres
📂 Đã load danh sách Validation Fold 4: 73 ca.
⚙️ Mode: VALIDATION SPLIT -> Found 73 cases (Filtered).

Index    | Case ID         | Dice WT  | Dice TC  | Dice ET  | Mean    
-------------------------------------------------------------------------------------
🔍 DEBUG: Đang tìm GT cho BRATS_001...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_001.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_001.nii -> KHÔNG
100% 8/8 [00:01<00:00,  5.65it/s]
1        | BRATS_001       | 0.9116   | 0.9281   | 0.8831   | 0.9076
    📸 Drawing Slice: 44
    ✅ Saved Snapshot: /content/drive/MyDrive/NCKH/nnUnet/inference_results/Dataset101_BraTS2020/EDLTra

##